# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/afreensumai64/ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# W03 setup — connect to the FlyRank warehouse

import os
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret
    (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FEB = f"{REL}/fact_content_daily_performance/month=2026-02/*.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"

print("DuckDB connection: READY")
print("February source: READY")
print("Content dimension: READY")

DuckDB connection: READY
February source: READY
Content dimension: READY


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature Vector

The feature vector uses information available at the February 2026 decision point.

The features are:

- `gsc_impressions` — total observed Google Search Console impressions during February 2026.
- `gsc_clicks` — total observed Google Search Console clicks during February 2026.
- `content_age_days` — number of days since the content creation date, measured as of February 28, 2026.

The feature vector contains numeric features only, so no categorical encoding is required.

Rows without a verified content creation date are excluded because content age cannot be calculated reliably for those rows.

Numeric missing values are filled using the median of the available feature values so that the resulting feature matrix is complete.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the feature vector from February decision-time information

FEB_FEATURES = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks
    FROM read_parquet('{FEB}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date
    FROM read_parquet('{DIM_CONTENT}')
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,
    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-02-28'
    ) AS content_age_days
FROM feb f
JOIN content c
    ON f.client_hash_id = c.client_hash_id
   AND f.content_hash_id = c.content_hash_id
WHERE c.content_created_date IS NOT NULL
""").df()

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "content_age_days"
]

feature_vector = FEB_FEATURES[feature_cols].copy()

# Numeric missing-value handling
for col in feature_cols:
    feature_vector[col] = pd.to_numeric(
        feature_vector[col],
        errors="coerce"
    )
    feature_vector[col] = feature_vector[col].fillna(
        feature_vector[col].median()
    )

print("Feature-vector rows:", len(feature_vector))
print("Feature columns:", feature_cols)
print("Missing values after filling:")
display(feature_vector.isna().sum())

display(feature_vector.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-vector rows: 153559
Feature columns: ['gsc_impressions', 'gsc_clicks', 'content_age_days']
Missing values after filling:


,0
gsc_impressions,0
gsc_clicks,0
content_age_days,0


,gsc_impressions,gsc_clicks,content_age_days
0,299.0,0.0,226
1,733.0,6.0,226
2,514.0,0.0,226
3,2931.0,3.0,226
4,970.0,2.0,226


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Notes

| Feature | Meaning | Missing-value handling | Available before prediction? |
|---|---|---|---|
| `gsc_impressions` | Observed Google Search Console impressions during February 2026 | Median fill after numeric conversion | Yes |
| `gsc_clicks` | Observed Google Search Console clicks during February 2026 | Median fill after numeric conversion | Yes |
| `content_age_days` | Days between content creation and February 28, 2026 | Rows without a creation date are excluded; remaining missing numeric values use median fill | Yes |

All three features are available at the February decision point.

No categorical features are used in this feature vector, so categorical encoding is not required.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature availability and missing-value audit

feature_notes = pd.DataFrame({
    "feature": [
        "gsc_impressions",
        "gsc_clicks",
        "content_age_days"
    ],
    "dtype": [
        str(feature_vector["gsc_impressions"].dtype),
        str(feature_vector["gsc_clicks"].dtype),
        str(feature_vector["content_age_days"].dtype)
    ],
    "missing_after_fill": [
        feature_vector["gsc_impressions"].isna().sum(),
        feature_vector["gsc_clicks"].isna().sum(),
        feature_vector["content_age_days"].isna().sum()
    ],
    "available_at_february_decision_point": [
        True,
        True,
        True
    ]
})

display(feature_notes)

assert feature_vector[feature_cols].isna().sum().sum() == 0

print("Feature availability check: PASSED")


,feature,dtype,missing_after_fill,available_at_february_decision_point
0,gsc_impressions,float64,0,True
1,gsc_clicks,float64,0,True
2,content_age_days,int64,0,True


Feature availability check: PASSED


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage Hunt

The main leakage risks for this project are:

1. Using March outcome information as a feature when the ranking decision is made at the end of February.
2. Using the outcome label itself, `went_dark`, as a feature.
3. Using product-generated fields such as priority or health scores that may already encode a downstream decision.
4. Using information that would not have been available at the February decision point.

The model therefore uses only February search signals and content age.

March data is reserved for constructing the observed evaluation outcome and is not included in the feature vector.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage audit

forbidden_feature_names = [
    "went_dark",
    "label",
    "target",
    "outcome",
    "priority_score",
    "health_score",
    "action_type",
    "recommended_action"
]

feature_name_check = [
    col for col in feature_vector.columns
    if any(term in col.lower() for term in forbidden_feature_names)
]

print("Forbidden feature-name matches:", feature_name_check)

assert feature_name_check == []

# Explicit future-window check
future_columns = [
    col for col in feature_vector.columns
    if "march" in col.lower()
    or "april" in col.lower()
    or "future" in col.lower()
]

print("Future-window feature matches:", future_columns)

assert future_columns == []

# Confirm the actual feature list
print("Final model features:", feature_cols)

assert feature_cols == [
    "gsc_impressions",
    "gsc_clicks",
    "content_age_days"
]

print("Leakage audit: PASSED")

Forbidden feature-name matches: []
Future-window feature matches: []
Final model features: ['gsc_impressions', 'gsc_clicks', 'content_age_days']
Leakage audit: PASSED


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded Fields

The following information was intentionally excluded from the feature vector:

- `went_dark` — this is the observed March outcome proxy, so using it as a February feature would leak the outcome into the model.
- March GSC performance — this occurs after the February decision point and is reserved for evaluation.
- Product-generated priority or health fields — these may encode downstream decisions and are not independent observable signals for this analysis.
- Client names, domains, URLs, and private search queries — these are not required for the public-safe analysis and should not appear in the deliverable.
- Credentials or access tokens — these are private and must never be included in notebook outputs or committed files.

The final feature vector therefore contains only three verified page-level numeric signals available at the February decision point.

In [5]:
# Final exclusion and privacy check

excluded_fields = pd.DataFrame({
    "field_or_category": [
        "went_dark",
        "March GSC performance",
        "Product-generated priority/health fields",
        "Client names, domains, URLs, private queries",
        "Credentials and access tokens"
    ],
    "reason_excluded": [
        "Observed outcome; would leak the target",
        "Future information relative to the February decision point",
        "May encode downstream decisions",
        "Not required for public-safe analysis",
        "Private security information"
    ]
})

display(excluded_fields)

print("Final feature vector:")
display(feature_vector[feature_cols].head())

print("Number of final features:", len(feature_cols))

,field_or_category,reason_excluded
0,went_dark,Observed outcome; would leak the target
1,March GSC performance,Future information relative to the February de...
2,Product-generated priority/health fields,May encode downstream decisions
3,"Client names, domains, URLs, private queries",Not required for public-safe analysis
4,Credentials and access tokens,Private security information


Final feature vector:


,gsc_impressions,gsc_clicks,content_age_days
0,299.0,0.0,226
1,733.0,6.0,226
2,514.0,0.0,226
3,2931.0,3.0,226
4,970.0,2.0,226


Number of final features: 3


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.